# Train MILK10k on Kaggle's free GPU

This notebook trains the **clinical EfficientNetV2-S, dermoscopic EfficientNetV2-S,
metadata MLP, and logistic-regression fusion**. It uses CUDA mixed precision, cached
preprocessing, pinned memory, persistent data-loader workers, and resumable runs.
With two GPUs, the image models train independently on GPUs 0 and 1.

**Before running:**
1. Upload this notebook to Kaggle. In Settings select the available GPU accelerator and enable Internet.
2. Upload `dist/skin_cancer_kaggle.zip` as a **private Kaggle Dataset** and attach it with Add Input.
3. Attach the **official MILK10k training metadata, ground-truth CSV, and unzipped image directory**.
   Other skin datasets are not interchangeable with MILK10k. The input-validator starter dataset is not used for disease training.
4. Run the cells in order. Use Save Version / Save & Run All and preserve outputs before ending the session.

All outputs go under `/kaggle/working`; attached datasets stay read-only.
The code bundle includes the trained input validator. No Kaggle account credentials are needed in this notebook.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil, zipfile, stat, importlib.util

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
PROJECT_DIR = WORK_ROOT / 'skin-cancer-project'
CODE_INPUT = None       # Optional explicit path to skin_cancer_kaggle.zip or a source folder.
RAW_OVERRIDES = {}      # Optional {official_filename_or_directory: '/kaggle/input/...'}.
RESUME_INPUT = None     # Optional previous training_artifacts.zip or extracted output folder.

BATCH_SIZE = 32         # Benchmark compares this with 16. Resume keeps the saved batch size.
NUM_WORKERS = 1         # Per image process; try 2 if the host has enough CPU capacity.
CACHE_WORKERS = 2
PARALLEL_IMAGE_MODELS = True  # Automatically falls back to sequential on one GPU.
RUN_CHECKS = True             # Includes a CUDA AMP smoke test before the long run.
BENCHMARK_STEPS = 10          # Set 0 to skip; does not save or change disease checkpoints.
CLINICAL_EPOCHS, DERMOSCOPIC_EPOCHS, METADATA_EPOCHS = 30, 20, 20
RUN_FUSION = True
SPLIT_SEED = 42
VAL_FRACTION, FUSION_FRACTION, HOLDOUT_FRACTION = 0.15, 0.10, 0.10

WORK_ROOT.mkdir(parents=True, exist_ok=True)


## Load the project into writable storage

Auto-detection expects one code bundle. If multiple releases are attached, set `CODE_INPUT` explicitly.
The bundle excludes your virtual environment, credentials, and training-image downloads.


In [ ]:
if CODE_INPUT is None:
    bundles = list(INPUT_ROOT.rglob('skin_cancer_kaggle.zip'))
    if len(bundles) == 1:
        CODE_INPUT = bundles[0]
    elif not bundles:
        sources = [p.parents[2] for p in INPUT_ROOT.rglob('train_model.py') if p.parent.name == 'scripts' and p.parent.parent.name == 'src']
        if len(sources) != 1:
            raise FileNotFoundError('Attach skin_cancer_kaggle.zip as an input or set CODE_INPUT to the project source folder.')
        CODE_INPUT = sources[0]
    else:
        raise ValueError('Multiple code bundles found. Set CODE_INPUT explicitly.')
CODE_INPUT = Path(CODE_INPUT)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
if CODE_INPUT.is_file():
    with zipfile.ZipFile(CODE_INPUT) as archive:
        for member in archive.infolist():
            target = (PROJECT_DIR / member.filename).resolve()
            if not target.is_relative_to(PROJECT_DIR.resolve()) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError(f'Unsafe archive entry: {member.filename}')
        for member in archive.infolist():
            target = PROJECT_DIR / member.filename
            if target.exists() and target.suffix in {'.pt', '.joblib'}:
                continue
            archive.extract(member, PROJECT_DIR)
else:
    for name in ('src', 'tests', 'notebooks'):
        source = CODE_INPUT / name
        target = PROJECT_DIR / name
        if not source.exists():
            continue
        if source.is_dir():
            # Preserve existing training checkpoints when rerunning setup.
            shutil.copytree(source, target, dirs_exist_ok=True, ignore=shutil.ignore_patterns('checkpoints', '__pycache__', '.torch'))
        else:
            shutil.copy2(source, target)
    validator_source = CODE_INPUT / 'src/checkpoints/input_validator.pt'
    validator_target = PROJECT_DIR / 'src/checkpoints/input_validator.pt'
    validator_target.parent.mkdir(parents=True, exist_ok=True)
    if validator_source.exists() and not validator_target.exists():
        shutil.copy2(validator_source, validator_target)
if not (PROJECT_DIR / 'src/scripts/train_kaggle.py').exists():
    raise RuntimeError('The attached source bundle is missing the Kaggle training scripts.')
sys.path.insert(0, str(PROJECT_DIR / 'src'))
os.environ['MILK10K_RAW_DIR'] = str(PROJECT_DIR / 'data/raw')
os.environ['MILK10K_PROCESSED_DIR'] = str(PROJECT_DIR / 'data/processed')
os.environ['MILK10K_CACHE_DIR'] = str(PROJECT_DIR / 'data/cache')
os.environ['TORCH_HOME'] = str(PROJECT_DIR / '.torch')
SRC = PROJECT_DIR / 'src'
CSV = PROJECT_DIR / 'data/processed/final_metadata_with_split.csv'
CACHE = PROJECT_DIR / 'data/cache'
ARTIFACT_ZIP = WORK_ROOT / 'training_artifacts.zip'
print('Writable project:', PROJECT_DIR)


## Check dependencies and GPU

Keep Kaggle's preinstalled CUDA PyTorch and torchvision. Do **not** install the Mac
local application dependencies here. Internet is needed for pretrained EfficientNet weights
unless you have already attached/cached them.


In [ ]:
required = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn', 'scipy': 'scipy', 'PIL': 'pillow', 'cv2': 'opencv-python-headless', 'joblib': 'joblib'}
if any(importlib.util.find_spec(module) is None for module in required):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *required.values()], check=True)
import torch, torchvision
version = tuple(int(part) for part in torch.__version__.split('+')[0].split('.')[:2])
if version < (2, 3):
    raise RuntimeError('Use a Kaggle GPU runtime with PyTorch >=2.3 for torch.amp.GradScaler.')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Enable GPU in Kaggle Settings and restart the session.')
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__)
for index in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(index)
    print(f'GPU {index}: {prop.name}, {prop.total_memory / 1024**3:.1f} GiB')
print('Two-image-model concurrency:', PARALLEL_IMAGE_MODELS and torch.cuda.device_count() >= 2)


In [ ]:
if RUN_CHECKS:
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=PROJECT_DIR, check=True)


## Link data and optionally restore a previous run

Required names: `MILK10k_Training_Metadata.csv`, `MILK10k_Training_GroundTruth.csv`,
and `MILK10k_Training_Input/` containing `IL_.../ISIC_....jpg` files.
Nested Kaggle dataset folders are discovered automatically. Compressed image archives
must be unzipped first; attach an extracted dataset or unzip into writable storage and set `RAW_OVERRIDES`.

For resuming, attach the prior notebook output. The helper automatically finds a unique
`training_artifacts.zip`; otherwise set `RESUME_INPUT`. Restore only your own trusted training artifacts.


In [ ]:
from utils.kaggle import link_raw_inputs, restore_artifacts, verify_split, export_artifacts
print(json.dumps(link_raw_inputs(INPUT_ROOT, PROJECT_DIR / 'data/raw', RAW_OVERRIDES), indent=2))
if RESUME_INPUT is None:
    previous = list(INPUT_ROOT.rglob('training_artifacts.zip'))
    if len(previous) == 1:
        RESUME_INPUT = previous[0]
    elif len(previous) > 1:
        raise ValueError('Multiple saved runs found; set RESUME_INPUT explicitly.')
if RESUME_INPUT is not None:
    restore_artifacts(RESUME_INPUT, PROJECT_DIR)

def run_script(name, *arguments):
    subprocess.run([sys.executable, '-u', str(SRC / 'scripts' / name), *map(str, arguments)], cwd=SRC, check=True)

if not (SRC / 'checkpoints/input_validator.pt').exists():
    raise FileNotFoundError('Attach the trained input_validator.pt in the code bundle before exporting an upload-ready pipeline.')


## Prepare patient/lesion-grouped splits

Approximately 65% train / 15% validation / 10% fusion fitting / 10% final holdout.
Fusion receives predictions from a split that the base models never trained on. The
holdout is used only for final fusion evaluation. This dedicated split is cheaper
than out-of-fold retraining, but leaves fewer examples for the base models.

Missing metadata is imputed by the encoder fitted on the training split. All 11
classes must be present in every split. Existing split CSVs are preserved on resume;
change fractions/seed only before starting a fresh run.


In [ ]:
if not (CSV.parent / 'final_metadata.csv').exists():
    run_script('prepare_data.py')
if not CSV.exists():
    run_script('make_splits.py', '--random-state', SPLIT_SEED,
               '--val-fraction', VAL_FRACTION, '--fusion-fraction', FUSION_FRACTION,
               '--holdout-fraction', HOLDOUT_FRACTION)
verify_split(CSV)


## Cache preprocessing once

This performs hair removal (dermoscopy), CLAHE, heuristic ROI cropping, and resizing
on CPU once. The lossless cache retains the same pixel values as online preprocessing;
random augmentation still runs during every training epoch. Cache keys include source
file identity and preprocessing code. Only source images are checked by the upload gate.


In [ ]:
run_script('cache_images.py', '--csv', CSV, '--cache-dir', CACHE, '--workers', CACHE_WORKERS)
# Download/verify pretrained weights once before starting concurrent image trainers.
from torchvision.models import EfficientNet_V2_S_Weights
weights = EfficientNet_V2_S_Weights.DEFAULT.get_state_dict(progress=True, check_hash=True)
del weights


## Benchmark batch size (optional)

Compare frozen and full fine-tuning steps on GPU 0 with AMP and channels-last. These
short benchmarks discard their weights. Select the faster full-fine-tuning batch that
fits, or set `BENCHMARK_STEPS=0` to use the configured batch size. Two GPUs share CPU
resources, so simultaneous full-run throughput can differ. Actual epoch times are logged.


In [ ]:
saved_runs = list((SRC / 'checkpoints').glob('*_resume.json'))
image_runs = [p for p in saved_runs if p.name.startswith(('clinical_', 'dermoscopic_'))]
if image_runs:
    saved_sizes = {json.loads(path.read_text())['config']['batch_size'] for path in image_runs}
    if len(saved_sizes) != 1:
        raise ValueError('Image checkpoints have different batch configurations; resume them separately.')
    BATCH_SIZE = saved_sizes.pop()
    print('Keeping saved batch size for resume:', BATCH_SIZE)
elif BENCHMARK_STEPS:
    benchmarks = []
    for candidate in sorted({16, BATCH_SIZE}):
        output = PROJECT_DIR / 'logs' / f'benchmark_batch_{candidate}.json'
        run_script('benchmark_training.py', '--csv', CSV, '--cache-dir', CACHE,
                   '--batch-size', candidate, '--steps', BENCHMARK_STEPS,
                   '--num-workers', NUM_WORKERS, '--output', output)
        report = json.loads(output.read_text())
        if report['fits'] and 'finetune' in report['phases']:
            benchmarks.append(report)
    if not benchmarks:
        raise RuntimeError('No benchmark batch fitted. Lower BATCH_SIZE below 16 and run a single candidate manually.')
    BATCH_SIZE = max(benchmarks, key=lambda report: report['phases']['finetune']['images_per_second'])['batch_size']
print('Training batch size:', BATCH_SIZE)


## Train and save artifacts

On two GPUs, clinical runs on `cuda:0` and dermoscopy on `cuda:1`. On one GPU they
run sequentially. Metadata trains afterward on CPU. Each epoch atomically saves the
best weights, optimizer, scheduler, AMP scaler, RNG state, configuration, and history.
Completed models are skipped on resume when data/configuration match.

If GPU memory runs out, start a fresh configuration with batch size 16 (or smaller).
If CPU loading is the bottleneck, compare sequential execution or adjust workers.

Fusion fitting uses no TTA for speed and evaluates once on the holdout. The explicit
`--trusted-benchmark` flag evaluates all curated benchmark images without filtering
through the upload gate; upload inference remains checked by default. Gate performance
must be evaluated separately, not by silently removing rejected benchmark examples.

The artifact ZIP is refreshed even if training raises an error. **Kaggle storage is
not permanent until outputs are saved**: preserve the ZIP/checkpoints using Save Version
with outputs enabled, or download them before the session ends. A forced session kill
can prevent the final ZIP refresh; individual epoch checkpoints remain the latest state
if you preserve the working directory outputs.


In [ ]:
try:
    run_script('train_kaggle.py', '--csv', CSV, '--cache-dir', CACHE,
               '--batch-size', BATCH_SIZE, '--num-workers', NUM_WORKERS,
               '--clinical-epochs', CLINICAL_EPOCHS, '--dermoscopic-epochs', DERMOSCOPIC_EPOCHS,
               '--metadata-epochs', METADATA_EPOCHS,
               '--parallel' if PARALLEL_IMAGE_MODELS else '--no-parallel', '--resume')
    if RUN_FUSION:
        run_script('train_fusion.py', '--csv', CSV, '--fit-split', 'fusion', '--eval-split', 'holdout',
                   '--no-tta', '--trusted-benchmark')
finally:
    export_artifacts(PROJECT_DIR, ARTIFACT_ZIP)


## Results and download

The ZIP contains best checkpoints, the metadata encoder, resume states, split CSVs,
per-epoch timing/metrics, logs, and (if enabled) fusion weights and holdout metrics.
Copy `src/checkpoints/` back to the matching local project to use the trained models.
Keep evaluation preprocessing/TTA consistent: this notebook fits fusion **without TTA**.

To continue in another session, attach these outputs and the original MILK10k input,
then rerun this notebook with the same epoch settings. Resume checks the dataset hash
and training configuration. Epoch seeds are restored, but persistent-worker augmentation
streams and CUDA kernels need not reproduce an uninterrupted run bit-for-bit.


In [ ]:
for path in sorted((SRC / 'checkpoints').glob('*_resume.json')):
    report = json.loads(path.read_text())
    print(path.name, 'complete=', report['complete'], 'epochs=', report['epoch'] + 1,
          'best validation score=', round(report['best_score'], 4))
fusion_metrics = SRC / 'checkpoints/best_fusion_lr.metrics.json'
if fusion_metrics.exists():
    print('Fusion holdout metrics:', fusion_metrics.read_text())
from IPython.display import FileLink, display
display(FileLink(str(ARTIFACT_ZIP)))
